# M13 lab solution: Embeddings & Model Selection

**Brief.** Embedding shoot-out: compare one hosted and two open-source models on a 30-question golden set and pick one, writing down the cost and latency per 1,000 chunks.

Run with the `hf` group: `uv run --group hf jupyter lab` (the open models run locally through sentence-transformers; the first run downloads them).

Open this notebook from `lab/solution/`, so `../data/` resolves to the lab's own data folder.

## Setup

In [ ]:
import json
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()  # API keys from the .env file at the repo root
sys.path.insert(0, str(Path("../../../../src").resolve()))

from lab_kit import baseline, hit_rate, is_hit, load_handbook, load_jsonl, mrr, recall_at_k  # noqa: E402

DATA = Path("../data/larkspur")
handbook = load_handbook(DATA / "handbook")
golden = load_jsonl(DATA / "golden.jsonl")
by_id = {g["id"]: g for g in golden}
scored = [g for g in golden if g["evidence_spans"]]
print(len(handbook), "handbook files,", len(golden), "golden questions,", len(scored), "with evidence")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunk_texts = [c for p in handbook for c in splitter.split_text(p["text"])]
print(len(chunk_texts), "chunks (the lab_kit.baseline split)")

In [ ]:
def texts(docs):
    return [d.page_content for d in docs]


def score(results, questions=None):
    """results: question id -> retrieved chunk texts in rank order; scored with lab_kit on the questions with evidence."""
    qs = questions or golden
    return {"hit_rate": round(hit_rate(results, qs), 3), "recall_at_k": round(recall_at_k(results, qs), 3),
            "mrr": round(mrr(results, qs), 3)}

## 1. Three models

One hosted model, `text-embedding-3-small` (OpenAI), and two open models run locally, `all-MiniLM-L6-v2` and `BAAI/bge-small-en-v1.5`, both with normalised vectors. The same model embeds the chunks and the questions in every row.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings

MODELS = {
    "text-embedding-3-small": OpenAIEmbeddings(model="text-embedding-3-small"),
    "all-MiniLM-L6-v2": HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",
                                              encode_kwargs={"normalize_embeddings": True}),
    "bge-small-en-v1.5": HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5",
                                               encode_kwargs={"normalize_embeddings": True}),
}

## 2. Embed, time and retrieve

Seconds per 1,000 chunks are measured on this corpus and scaled (`seconds / chunks * 1000`); the open models' figure depends on this machine. Cost per 1,000 chunks applies only to the hosted model: tokens are counted with `tiktoken`'s `cl100k_base` encoding, the one the `text-embedding-3` models use, times the price per million tokens set below.

In [ ]:
import numpy as np
import tiktoken

# USD per million tokens for text-embedding-3-small. Confirm it on the OpenAI price page on the day you run
# the lab and record that date in the README.
PRICE_PER_1M = 0.02
tokens = sum(len(t) for t in tiktoken.get_encoding("cl100k_base").encode_batch(chunk_texts))

K = 4
rows = []
for name, model in MODELS.items():
    t0 = time.perf_counter()
    doc_vecs = np.array(model.embed_documents(chunk_texts))
    seconds = time.perf_counter() - t0
    q_vecs = np.array(model.embed_documents([g["question"] for g in golden]))
    sims = q_vecs @ doc_vecs.T / np.outer(np.linalg.norm(q_vecs, axis=1), np.linalg.norm(doc_vecs, axis=1))
    results = {g["id"]: [chunk_texts[i] for i in np.argsort(-sims[n])[:K]] for n, g in enumerate(golden)}
    hosted = name == "text-embedding-3-small"
    rows.append({"model": name, "dims": doc_vecs.shape[1], **score(results),
                 "s_per_1k_chunks": round(seconds / len(chunk_texts) * 1000, 2),
                 "usd_per_1k_chunks": round(tokens / len(chunk_texts) * 1000 * PRICE_PER_1M / 1e6, 6) if hosted else 0.0})
    print(rows[-1])

## Results and the pick

In [ ]:
import pandas as pd

table = pd.DataFrame(rows)
print("run:", time.strftime("%Y-%m-%d"), "| corpus:", len(chunk_texts), "chunks,", tokens, "tokens")
print(table.to_string(index=False))

# The rule: best hit rate; ties go to recall, then MRR, then the cheaper and faster model.
pick = table.sort_values(["hit_rate", "recall_at_k", "mrr", "usd_per_1k_chunks", "s_per_1k_chunks"],
                         ascending=[False, False, False, True, True]).iloc[0]
runner = table[table["model"] != pick["model"]].sort_values("hit_rate", ascending=False).iloc[0]
print(f"\nPick: {pick['model']}. Hit rate {pick['hit_rate']} vs {runner['hit_rate']} for {runner['model']}, "
      f"recall {pick['recall_at_k']}, {pick['dims']} dimensions, {pick['s_per_1k_chunks']} s and "
      f"{pick['usd_per_1k_chunks']} USD per 1,000 chunks.")

## What this shows

- Domain fit is measured on your own golden set, not read off a leaderboard; the table is that measurement for the handbook.
- Dimensions are a cost paid on every query and every stored vector: compare the `dims` column with the quality columns before paying for the larger vector.
- The hosted model's cost per 1,000 chunks is small next to the time column for the local models on a laptop; for a fixed corpus, the cheap option can be the hosted one. The pick line states the rule it used, so a different rule gives a different, equally reproducible pick.